# GPReg Demo Notebook

This notebook walks through the main features of the GPReg package:

1. Basic 1D regression with the exact GP
2. Composite kernels and how to read them
3. Multivariate input + DataFrame with categoricals
4. Diagnostics: RMSE, NLPD, LOO-CV, calibration
5. Sparse GP for large datasets
6. PyTorch backend for hyperparameter optimization

Run cells top-to-bottom.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gpreg import (
    GaussianProcessRegressor, SparseGPRegressor,
    RBF, Matern, Linear, White,
    StandardScaler, CategoricalEncoder, PCA,
    make_gp_pipeline, Pipeline,
    rmse, nlpd, loo_cv, calibration_curve,
    save_model, load_model,
)
from gpreg.diagnostics import (
    plot_predictions_1d, plot_predictions_2d,
    plot_calibration, plot_residuals,
    plot_kernel_heatmap, plot_pair,
)

np.random.seed(42)

## 1. Basic 1D regression

Fit a GP to noisy samples from sin(x) + 0.5cos(0.3x). The GP recovers the underlying smooth function and supplies uncertainty estimates that grow outside the training range.

In [ ]:
X = np.linspace(-3, 3, 30).reshape(-1, 1)
y_true = lambda x: np.sin(x) + 0.5 * np.cos(0.3 * x)
y = y_true(X.ravel()) + 0.15 * np.random.randn(30)

gp = GaussianProcessRegressor(
    kernel=RBF(length_scale=1.0) + White(noise_var=0.1),
    n_restarts=3, random_state=0,
)
gp.fit(X, y)

print('Fitted kernel:', gp.kernel)
print(f'Log marginal likelihood: {gp.log_marginal_likelihood_:.3f}')

X_test = np.linspace(-4, 4, 200).reshape(-1, 1)
fig, ax = plot_predictions_1d(gp, X_test, n_samples=3, true_fn=y_true, random_state=0)
plt.show()

## 2. Composite kernels

Kernels can be added (`+`) and multiplied (`*`) to express richer covariance structures. Below: Linear + RBF captures a global trend with local fluctuations.

In [ ]:
X = np.linspace(0, 10, 60).reshape(-1, 1)
y = 0.5 * X.ravel() + np.sin(X.ravel()) + 0.2 * np.random.randn(60)

trend_kernel = Linear(signal_var=1.0) + RBF(length_scale=1.0) + White(0.1)
gp = GaussianProcessRegressor(kernel=trend_kernel, n_restarts=3, random_state=0)
gp.fit(X, y)

print('Fitted kernel:', gp.kernel)

X_test = np.linspace(-1, 12, 200).reshape(-1, 1)
fig, ax = plot_predictions_1d(gp, X_test)
ax.set_title('Linear + RBF kernel: linear trend with periodic-like structure')
plt.show()

## 3. Multivariate numeric inputs (2D)

With two numeric features, we can visualize the predictive surface as a heatmap. The package handles arbitrary input dimensionality — 2D is just easy to look at.

In [ ]:
n = 80
x1 = np.random.uniform(-3, 3, n)
x2 = np.random.uniform(-3, 3, n)
X_2d = np.column_stack([x1, x2])
y_2d = np.exp(-(x1 ** 2 + x2 ** 2) / 4) * np.sin(2 * x1) + 0.1 * np.random.randn(n)

# Pipeline: scale the inputs (important when features have different ranges), then GP
from gpreg.preprocessing import StandardScaler, Pipeline
pipe_2d = Pipeline([
    ('scale', StandardScaler()),
    ('gp', GaussianProcessRegressor(kernel=RBF() + White(0.1), n_restarts=3, random_state=0)),
])
pipe_2d.fit(X_2d, y_2d)

print(f'Fitted kernel: {pipe_2d.estimator.kernel}')
print(f'Training R^2: {pipe_2d.score(X_2d, y_2d):.4f}')

# Visualize predictive mean and std over a 2D grid
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_predictions_2d(pipe_2d.estimator, x1_range=(-3, 3), x2_range=(-3, 3),
                    n_grid=40, ax=axes[0], plot_type='mean')
# Training data also needs to be scaled for the underlying estimator's plot;
# we transform manually since plot_predictions_2d reads the scaled X_train_
axes[0].set_title('Predictive mean (scaled space)')
plot_predictions_2d(pipe_2d.estimator, x1_range=(-3, 3), x2_range=(-3, 3),
                    n_grid=40, ax=axes[1], plot_type='std')
axes[1].set_title('Predictive std (scaled space)')
plt.tight_layout()
plt.show()

## 4. Multivariate inputs with categoricals

Real datasets are usually DataFrames with mixed types. The pipeline handles dummy coding, scaling, and the GP fit transparently.

In [ ]:
n = 200
df = pd.DataFrame({
    'temperature': np.random.uniform(10, 30, n),
    'humidity': np.random.uniform(0.2, 0.9, n),
    'season': np.random.choice(['spring', 'summer', 'fall', 'winter'], n),
})
season_effect = df['season'].map({'spring': 1.0, 'summer': 2.5, 'fall': 0.5, 'winter': -1.0})
df['y'] = (
    0.1 * df['temperature']
    - 1.5 * df['humidity']
    + season_effect
    + 0.2 * np.random.randn(n)
)

idx = np.random.permutation(n)
tr, te = idx[:150], idx[150:]
df_tr, df_te = df.iloc[tr], df.iloc[te]

pipeline = make_gp_pipeline(
    GaussianProcessRegressor(kernel=RBF() + White(0.1), n_restarts=3, random_state=0)
)
pipeline.fit(df_tr[['temperature', 'humidity', 'season']], df_tr['y'].values)

y_pred, y_std = pipeline.predict(df_te[['temperature', 'humidity', 'season']], return_std=True)
y_te = df_te['y'].values

print(f'Test RMSE: {rmse(y_te, y_pred):.4f}')
print(f'Test NLPD: {nlpd(y_te, y_pred, y_std):.4f}')
print(f'Test R^2:  {pipeline.score(df_te[["temperature", "humidity", "season"]], y_te):.4f}')
print(f'Categoricals encoded: {pipeline.named_steps["encode"].feature_names_out_}')

## 5. Multivariate outputs

Real problems often have multiple targets. `MultiOutputGP` wraps any single-output GP to predict several output dimensions in parallel — one independent GP per output. Each output gets its own optimized kernel hyperparameters, which lets the model adapt to outputs of different smoothness.

In [ ]:
from gpreg import MultiOutputGP

# Three different functions of the same input
X = np.linspace(-3, 3, 40).reshape(-1, 1)
Y = pd.DataFrame({
    'sin':      np.sin(X).ravel(),
    'cos':      np.cos(X).ravel(),
    'parabola': 0.5 * X.ravel() ** 2 - 1.0,
}) + 0.1 * np.random.randn(40, 3)

base = GaussianProcessRegressor(kernel=RBF() + White(0.1), n_restarts=2, random_state=0)
mgp = MultiOutputGP(base)
mgp.fit(X, Y)

X_test = np.linspace(-4, 4, 200).reshape(-1, 1)
Y_mean, Y_std = mgp.predict(X_test, return_std=True)
print(f'Predicted shape: mean={Y_mean.shape}, std={Y_std.shape}')
print()
for name, gp in zip(mgp.output_names_, mgp.estimators_):
    print(f'{name:9s} -> {gp.kernel}')

# Plot each output independently
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for j, (name, ax) in enumerate(zip(mgp.output_names_, axes)):
    ax.fill_between(X_test.ravel(),
                    Y_mean[:, j] - 2 * Y_std[:, j],
                    Y_mean[:, j] + 2 * Y_std[:, j],
                    alpha=0.25, color='C0')
    ax.plot(X_test.ravel(), Y_mean[:, j], color='C0', lw=2, label='GP mean')
    ax.scatter(X.ravel(), Y[name], color='black', s=20, zorder=5, label='Data')
    ax.set_title(f'Output: {name}')
    ax.set_xlabel('x')
    ax.set_ylabel(name)
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nNote how each output got a different length-scale — the model adapts')
print(f'separately to the smoothness of each target.')

## 6. Diagnostics: calibration and residuals

RMSE and NLPD give scalar summaries; the calibration plot tells you whether the GP's confidence intervals contain the true value at the rate they claim to.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_calibration(y_te, y_pred, y_std, ax=axes[0])
plot_residuals(y_te, y_pred, y_std, ax=axes[1])
plt.tight_layout()
plt.show()

loo = loo_cv(pipeline.estimator)
print(f'LOO-CV RMSE: {loo["loo_rmse"]:.4f}')
print(f'LOO-CV NLPD: {loo["loo_nlpd"]:.4f}')

## 7. Sparse GP for large datasets

Exact GP cost is O(n³). At n=2000 it gets uncomfortable; at n=10000 it's impractical. The FITC sparse GP uses m << n inducing points and brings the cost down to O(nm²).

In [ ]:
import time
n = 1500
X = np.random.uniform(-5, 5, n).reshape(-1, 1)
y = np.sin(1.5 * X).ravel() + 0.5 * np.cos(0.3 * X).ravel() + 0.2 * np.random.randn(n)

# Exact
t0 = time.time()
exact = GaussianProcessRegressor(kernel=RBF()+White(0.1), n_restarts=0, random_state=0)
exact.fit(X, y)
t_exact = time.time() - t0

# Sparse
t0 = time.time()
sparse = SparseGPRegressor(kernel=RBF()+White(0.1), n_inducing=50, n_restarts=0, random_state=0)
sparse.fit(X, y)
t_sparse = time.time() - t0

print(f'Exact GP fit time:  {t_exact:.2f}s')
print(f'Sparse GP fit time: {t_sparse:.2f}s ({t_exact / t_sparse:.1f}x speedup)')

X_plot = np.linspace(-5, 5, 300).reshape(-1, 1)
fig, ax = plot_predictions_1d(sparse, X_plot)
y_lim = ax.get_ylim()
ax.scatter(sparse.Z_.ravel(), [y_lim[0]]*len(sparse.Z_),
           marker='|', color='red', s=200, linewidths=2, label='Inducing points')
ax.legend()
plt.show()

## 8. PyTorch backend

Optional: use PyTorch's autograd instead of SciPy's L-BFGS-B for hyperparameter optimization. Useful when you've defined a custom kernel that's hard to differentiate analytically. Both should converge to the same answer.

In [ ]:
X = np.linspace(-3, 3, 40).reshape(-1, 1)
y = np.sin(X).ravel() + 0.1 * np.random.randn(40)

gp_scipy = GaussianProcessRegressor(kernel=RBF()+White(0.1), n_restarts=2, random_state=0)
gp_scipy.fit(X, y)

gp_torch = GaussianProcessRegressor(
    kernel=RBF()+White(0.1), optimizer='pytorch', n_restarts=2, random_state=0
)
gp_torch.fit(X, y)

print('SciPy backend:  ', gp_scipy.kernel)
print('PyTorch backend:', gp_torch.kernel)

## 9. Save and load

Pickle a fitted model and reload it later — predictions should be exactly preserved.

In [ ]:
save_model(gp_scipy, '/tmp/my_gp.pkl')
loaded = load_model('/tmp/my_gp.pkl')
X_test = np.linspace(-4, 4, 50).reshape(-1, 1)
print('Predictions identical after round-trip:',
      np.allclose(gp_scipy.predict(X_test), loaded.predict(X_test)))